# STS-B cosine-threshold sweep

This notebook loads the GLUE STS-B validation split from Hugging Face, binarizes human similarity scores using `score >= 4`, embeds both sentences with the configured FastEmbed model, and reports measured cosine-only precision/recall/FPR across thresholds. It reports cosine computation latency separately from the embedding batch. No benchmark results are checked into this notebook; run the cells to obtain results for your environment.

In [1]:
import numpy as np
from datasets import load_dataset

from aegis.engine.embeddings import FastEmbedder
from benchmarks.evaluation import (
    PairExample,
    stsb_binary_labels,
)

/home/ray/aegis-gateway/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("nyu-mll/glue", "stsb", split="validation")
human_scores = [float(row["label"]) for row in dataset]
labels = stsb_binary_labels(human_scores)
examples = [
    PairExample(
        candidate=str(row["sentence1"]),
        query=str(row["sentence2"]),
        label=label,
    )
    for row, label in zip(dataset, labels, strict=True)
]
len(examples), sum(labels)

(1500, 264)

In [3]:
from importlib import reload

import benchmarks.evaluation as evaluation

evaluation = reload(evaluation)
embedder = FastEmbedder()
try:
    (
        pair_scores,
        embedding_seconds,
        cosine_seconds,
    ) = await evaluation.score_pair_cosines(
        examples,
        embedder,
    )
finally:
    await embedder.close()

similarities = [score.cosine_similarity for score in pair_scores]
thresholds = np.linspace(-1.0, 1.0, 101).tolist()
results = evaluation.threshold_sweep(labels, similarities, thresholds)
print(f"Embedding batch seconds: {embedding_seconds:.6f}")
print(
    f"Cosine scoring mean milliseconds per pair: {np.mean(cosine_seconds) * 1000:.6f}"
)

Embedding batch seconds: 48.724101
Cosine scoring mean milliseconds per pair: 0.007605


In [4]:
display(results)

[{'threshold': -1.0,
  'true_positive': 264,
  'true_negative': 0,
  'false_positive': 1236,
  'false_negative': 0,
  'precision': 0.176,
  'recall': 1.0,
  'false_positive_rate': 1.0,
  'accuracy': 0.176},
 {'threshold': -0.98,
  'true_positive': 264,
  'true_negative': 0,
  'false_positive': 1236,
  'false_negative': 0,
  'precision': 0.176,
  'recall': 1.0,
  'false_positive_rate': 1.0,
  'accuracy': 0.176},
 {'threshold': -0.96,
  'true_positive': 264,
  'true_negative': 0,
  'false_positive': 1236,
  'false_negative': 0,
  'precision': 0.176,
  'recall': 1.0,
  'false_positive_rate': 1.0,
  'accuracy': 0.176},
 {'threshold': -0.94,
  'true_positive': 264,
  'true_negative': 0,
  'false_positive': 1236,
  'false_negative': 0,
  'precision': 0.176,
  'recall': 1.0,
  'false_positive_rate': 1.0,
  'accuracy': 0.176},
 {'threshold': -0.92,
  'true_positive': 264,
  'true_negative': 0,
  'false_positive': 1236,
  'false_negative': 0,
  'precision': 0.176,
  'recall': 1.0,
  'false_posi